# Lab 07-11 · Three-Phase Circuits

Companion to chapter [07-11](../../chapters/07-11-three-phase-circuits.md). Handout: [README.md](README.md).

Circuits here are solved whole, with all three sources, rather than by the per-phase shortcut, and powers are computed from sampled waveforms, so the chapter's $\sqrt3$ relations and power formulas are being checked, not assumed. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import cmath
import math
import shutil
import subprocess
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)
a = cmath.exp(-2j * math.pi / 3)
deg = lambda z: math.degrees(cmath.phase(z))


def ngspice_values(path):
    out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) >= 2:
            try:
                vals[parts[0].strip()] = float(parts[1].split()[0])
            except (ValueError, IndexError):
                pass
    return vals

ngspice available: True


## Part 1 — Phasors

**🔮 Predict:** the angle from $\mathbf{V}_{an}$ to $\mathbf{V}_{ab}$, and from a Δ phase current to its line current.

In [2]:
Vp = 230.94
Van, Vbn, Vcn = Vp, Vp * a, Vp * a.conjugate()
Vab = Van - Vbn
print(f"V_ab = {abs(Vab):.2f} V at {deg(Vab):.2f} deg (sqrt3 x {Vp} = {math.sqrt(3) * Vp:.2f})")
Z = 8 + 6j
Iab, Ica = Vab / Z, (Vcn - Van) / Z
Ia = Iab - Ica
print(f"Delta: phase current {abs(Iab):.2f} A at {deg(Iab):.2f}, line current {abs(Ia):.2f} A at {deg(Ia):.2f} (lags by {deg(Iab) - deg(Ia):.2f} deg)")
assert abs(deg(Vab) - 30) < 1e-9 and abs(deg(Iab) - deg(Ia) - 30) < 1e-9

V_ab = 400.00 V at 30.00 deg (sqrt3 x 230.94 = 400.00)
Delta: phase current 40.00 A at -6.87, line current 69.28 A at -36.87 (lags by 30.00 deg)


## Part 2 — Constant power

**🔮 Predict:** the swing of a single-phase 12.8 kW load's instantaneous power at power factor 0.8, and of the balanced three-phase total.

In [3]:
t = np.linspace(0, 0.04, 8001)
w = 2 * math.pi * 50
Vm, Im, th = Vp * math.sqrt(2), abs(Vp / Z) * math.sqrt(2), cmath.phase(Z)
phases = [Vm * np.cos(w * t - k * 2 * math.pi / 3) * Im * np.cos(w * t - k * 2 * math.pi / 3 - th) for k in range(3)]
total = sum(phases)
S1 = 12.8e3 / 0.8
single = math.sqrt(2) * np.cos(w * t) * math.sqrt(2) * S1 * np.cos(w * t - math.acos(0.8))
print(f"single phase: {single.min() / 1e3:.2f} to {single.max() / 1e3:.2f} kW; three-phase total: {total.min() / 1e3:.4f} to {total.max() / 1e3:.4f} kW")
assert np.ptp(total) < 1e-6 * total.mean()
if HAVE_NGSPICE:
    print("ngspice:", ngspice_values("tran_power.cir"))

single phase: -3.20 to 28.80 kW; three-phase total: 12.8000 to 12.8000 kW
ngspice: {'Doing analysis at TEMP': 27.0, 'pmin': 12799.99, 'pmax': 12799.99}


## Part 3 — The neutral

**🔮 Predict** (the chapter's guided exercise): with 10, 20 and 40 Ω on a 230/400 V supply and the neutral broken, which phase's load voltage rises most?

In [4]:
Y = np.array([1 / 10, 1 / 20, 1 / 40])
Vs = np.array([Vp * 230 / Vp, 230 * a, 230 * a.conjugate()])
In = sum(Vs * Y)
print(f"neutral connected: neutral current {abs(In):.2f} A at {deg(In):.2f} deg")
Vn = sum(Vs * Y) / Y.sum()                 # star point with the neutral open (Millman)
for name, V in zip("abc", Vs):
    print(f"neutral open: phase {name} load voltage {abs(V - Vn):.1f} V")
print(f"star point displaced by {abs(Vn):.2f} V")
assert abs(abs(Vs[2] - Vn) - 301.1) < 0.05
# third harmonics in the neutral, by sampling
ts = np.linspace(0, 0.02, 20000, endpoint=False)
i3 = sum(10 * math.sqrt(2) * np.cos(3 * (w * ts - k * 2 * math.pi / 3)) for k in range(3))
print(f"neutral RMS from 10 A third harmonic per phase: {np.sqrt(np.mean(i3**2)):.2f} A")
if HAVE_NGSPICE:
    print("ngspice:", ngspice_values("ac_three_phase.cir"))

neutral connected: neutral current 15.21 A at -19.11 deg
neutral open: phase a load voltage 150.6 V
neutral open: phase b load voltage 260.8 V
neutral open: phase c load voltage 301.1 V
star point displaced by 86.93 V
neutral RMS from 10 A third harmonic per phase: 30.00 A


ngspice: {'Doing analysis at TEMP': 27.0, 'ia1': 23.09401, 'pa1': -36.8699, 'ptot': 12800.0, 'in2': 15.21307, 'pn2': -19.1066, 'w1': 3628.719, 'w2': 9171.282, 'va3': 150.5703, 'vb3': 260.7955, 'vc3': 301.1407, 'vn3': 86.93183}


## Part 4 — Two wattmeters

**🔮 Predict:** how the readings change as the power factor falls from 1 to 0.3.

In [5]:
VL, IL = 400.0, 23.09
for pf in (1.0, 0.9, 0.8, 0.6, 0.5, 0.4, 0.3):
    th = math.degrees(math.acos(pf))
    W1 = VL * IL * math.cos(math.radians(30 + th)); W2 = VL * IL * math.cos(math.radians(30 - th))
    print(f"PF {pf:.1f}: W1 = {W1 / 1e3:6.2f} kW, W2 = {W2 / 1e3:6.2f} kW, sum = {(W1 + W2) / 1e3:6.2f} kW")
assert abs(VL * IL * math.cos(math.radians(30 + 60))) < 1e-9

PF 1.0: W1 =   8.00 kW, W2 =   8.00 kW, sum =  16.00 kW
PF 0.9: W1 =   5.19 kW, W2 =   9.21 kW, sum =  14.40 kW
PF 0.8: W1 =   3.63 kW, W2 =   9.17 kW, sum =  12.80 kW
PF 0.6: W1 =   1.10 kW, W2 =   8.49 kW, sum =   9.60 kW
PF 0.5: W1 =   0.00 kW, W2 =   8.00 kW, sum =   8.00 kW
PF 0.4: W1 =  -1.03 kW, W2 =   7.43 kW, sum =   6.40 kW
PF 0.3: W1 =  -2.01 kW, W2 =   6.80 kW, sum =   4.80 kW


## Your turn

1. A balanced Y load of $20\angle30°$ Ω per phase is on a 480 V supply. Find the line current and the total power.
2. A Δ load draws 30 A line current. What current flows in each Δ element?
3. Two wattmeters read 6 kW and 6 kW. What are $P$, $Q$ and the power factor?

<details><summary>Answers</summary>

1. $V_p = 480/\sqrt3$ = 277.1 V, $I = 277.1/20$ = **13.86 A**; $P = 3\times277.1\times13.86\times\cos30°$ = **9.98 kW**.
2. $30/\sqrt3$ = **17.32 A**.
3. $P$ = **12 kW**, $Q$ = **0**, power factor **1**.

</details>

In [6]:
Vp1 = 480 / math.sqrt(3); I1 = Vp1 / 20
print(f"1) {I1:.2f} A, {3 * Vp1 * I1 * math.cos(math.radians(30)) / 1e3:.2f} kW")
print(f"2) {30 / math.sqrt(3):.2f} A")
assert abs(3 * Vp1 * I1 * math.cos(math.radians(30)) - 9976.6) < 1

1) 13.86 A, 9.98 kW
2) 17.32 A
